In [1]:
import os

from app.rag.embeddings import create_embedding

os.environ["DATABASE_URL"] = (
    "postgresql+psycopg://"
    "document_agent:document_agent@127.0.0.1:5432/document_agent"
)

from app.db import engine

from sqlalchemy import text

text = "Автомобиль движется по дороге."

embedding = create_embedding(text)

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
1024
[0.013724351, -0.010386699, -0.00018689301, 0.00757554, -0.016950255, -0.006857085, 0.012174972, 0.05390713, -0.042099312, -0.0059850896]


In [2]:
import os

from sqlmodel import Session, select

os.environ["DATABASE_URL"] = (
    "postgresql+psycopg://"
    "document_agent:document_agent@127.0.0.1:5432/document_agent"
)

from app.db import engine
from app.models import Document

with Session(engine) as session:
    documents = session.exec(
        select(Document).order_by(Document.id)
    ).all()

    for doc in documents:
        print(doc.id, doc.title, doc.path)

1 Embedding test internal://embedding-test
2 README data/raw/pgvector/README.md
3 README data/raw/pgvector/README.md


In [10]:
document_id = documents[2].id

In [11]:
from app.rag.embeddings import embed_document

with Session(engine) as session:
    embed_document(session, document_id)

In [12]:
from app.models import DocumentChunk

with Session(engine) as session:
    chunks = session.exec(
        select(DocumentChunk)
        .where(DocumentChunk.document_id == document_id)
        .order_by(DocumentChunk.chunk_index)
    ).all()

    print("Chunks:", len(chunks))
    print("Embeddings:", sum(chunk.embedding is not None for chunk in chunks))
    print("Dimensions:", len(chunks[0].embedding))

Chunks: 93
Embeddings: 93
Dimensions: 1024


In [13]:
from sqlmodel import Session, select

from app.db import engine
from app.models import Document, DocumentChunk

with Session(engine) as session:
    documents = session.exec(
        select(Document).order_by(Document.id)
    ).all()

    for doc in documents:
        chunks_count = len(
            session.exec(
                select(DocumentChunk)
                .where(DocumentChunk.document_id == doc.id)
            ).all()
        )

        print(
            f"id={doc.id} | "
            f"title={doc.title!r} | "
            f"chunks={chunks_count}"
        )

id=1 | title='Embedding test' | chunks=3
id=2 | title='README' | chunks=93
id=3 | title='README' | chunks=93
